# Sidelink Mode 2 in 5G-LENA v2x and in MoReV2X

Both simulators implement the same procedure, TS 38.214 section 8.1.4, and the same SPS rules, TS 38.321 section 5.22.1. They differ in what is around it, and in which parts of the standard they cover.

- **5G-LENA v2x**: the `nr` module, branch `v2x`, on the CTTC `ns-3-dev` tree at `ns-3-dev-v2x-v1.1`, based on ns-3.42. This is your groupcast base. The description below is of the pinned commit `32156601`. Your `patches/nr/local.patch` is covered [separately](#your-local-patch).
- **MoReV2X**: this repository.

[Notebook 02](02_mode2_resource_allocation.ipynb) has MoReV2X in detail, so here it is mainly the comparison.

## In short

| | 5G-LENA v2x | MoReV2X |
|---|---|---|
| Focus | Full NR stack, with sidelink as one mode | Mode 2 scheduling studies |
| Scheduler | **Pluggable class** `NrSlUeMacScheduler`, set per UE with `NrSlHelper::SetNrSlSchedulerTypeId` | One function with flags |
| Traffic types | Broadcast, groupcast, unicast | Broadcast only |
| HARQ | **PSFCH feedback**, or blind | Blind only, 2 transmissions at most |
| Transmissions per TB | Up to `slMaxTxTransNumPssch` | 2 |
| Priority | In LC selection, not in the RSRP threshold | None |
| **Re-evaluation** | **None** | **Yes**, with 2 variants |
| Dynamic grants | Per logical channel | Per UE, `Cresel = 1` |
| Adaptive reservation | None | `--AdaptiveRRI`, not 3GPP |
| TDD and slot pattern | Modelled: `slTimeResources` bitmap and TDD pattern | Every slot is available |
| Channel | Full 3GPP model with fast fading, beams, antennas | Pathloss, log-normal and NLOSv shadowing, per TB BLER curves |
| Error model | EESM with HARQ combining | BLER curves from link-level tables |
| SUMO | `src/traci` in your tree | `src/traci`, same origin |

## How 5G-LENA v2x does it

### 1. Configuration

Most Mode 2 parameters are in a **resource pool**, not on the MAC. From `examples/nr-v2x-examples/cttc-nr-v2x-demo-simple.cc`:

```cpp
Ptr<NrSlCommResourcePoolFactory> ptrFactory = Create<NrSlCommResourcePoolFactory> ();
std::vector<std::bitset<1>> slBitmap = {1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1};
ptrFactory->SetSlTimeResources (slBitmap);   // which slots belong to the pool
ptrFactory->SetSlSensingWindow (100);        // T0 in ms
ptrFactory->SetSlSelectionWindow (5);
ptrFactory->SetSlSubchannelSize (50);
ptrFactory->SetSlMaxNumPerReserve (3);       // N_MAX: resources one SCI can point to
ptrFactory->SetSlResourceReservePeriodList ({0, 100});  // allowed RRIs
```

Grants need the slot grid first: the slot must be uplink in the TDD pattern **and** set in the bitmap. MoReV2X has no such filter, so every slot is a candidate.

The rest is on `NrSlUeMac`:

| Attribute | Default | Meaning |
|---|---|---|
| `EnableSensing` | false | **Off by default**: selection is then random over all candidates |
| `T1`, `T2` | 2, 33 slots | Selection window. `T2` is capped by the PDB |
| `Tproc0` | 1 | Sensing results younger than this are ignored |
| `SlThresPsschRsrp` | -128 dBm | Starting threshold |
| `ResourcePercentage` | 20 | The X of step 7 |
| `MinTimeGapProcessing` | 2 | Slots to prepare a re-transmission |

### 2. Trigger

`NrSlUeMac::DoSlotIndication` calls `SchedNrSlTriggerReq` each slot. The scheduler then:

1. **`TxResourceReselectionCheck`** decides which logical channels need a grant. For SPS, the check follows 38.321:

```cpp
if (itGrantFoundLc->slResoReselCounter == 0)
{
    double randProb = m_ueSelectedUniformVariable->GetValue (0, 1);
    if (GetMac ()->GetSlProbResourceKeep () > randProb)
        // keep the grant, restart slResoReselCounter
```

2. **`LogicalChannelPrioritization`** picks a destination and channels. Priority matters here, and `PriorityToSps` breaks a tie between SPS and dynamic.
3. **`AttemptGrantAllocation`** asks the MAC for candidates, then builds the grant.

### 3. Sensing

The PHY builds a `SensingData` from each decoded SCI 1-A and passes it through `ReceiveSensingData`:

```cpp
SensingData (SfnSf sfn, uint16_t rsvp, uint8_t sbChLength, uint8_t sbChStart, uint8_t prio,
             double slRsrp, uint8_t gapReTx1, uint8_t sbChStartReTx1,
             uint8_t gapReTx2, uint8_t sbChStartReTx2)
```

It holds the priority and up to 2 future re-transmissions, as SCI 1-A does in the standard.

### 4. Selection

`NrSlUeMac::GetCandidateResourcesPrivate` follows the steps of TS 38.214, and the code names them.

**T2.** With a PDB: `min(PDB, T2)`. Without one: `max(T2, T2min)`.

**Step 4.** `GetNrSlCommOpportunities` lists the pool slots in `[n + T1, n + T2]`, and `GetCandidateResourcesFromSlots` expands them into subchannels. This is `S_A`, of size `M_total`.

If `EnableSensing` is false, it stops here.

**Step 5.** `ExcludeResourcesBasedOnHistory` removes slots of the UE's own past transmissions, projected over every RRI in the pool list. **If fewer than X % remain, the step is undone**:

```cpp
if (remainingCandidates.size () >= (GetResourcePercentage () / 100.0) * mTotal) { /* keep */ }
else { remainingCandidates = candidateResources; }
```

**Step 6.** Projects each sensed SCI into the window, and drops a candidate if any of its `cResel` future transmissions overlaps one in time and frequency, above the threshold:

```cpp
if (itFutureCand.sfn.Normalize () == itReservedResourceProjection.sfn.Normalize ())
    if (itReservedResourceProjection.slRsrp > rsrpThreshold)
        if (OverlappedResource (...))
            itCandidate = remainingCandidates.erase (itCandidate);
```

**Step 7.** Raises the threshold by 3 dB and repeats until X % remain. Past 0 dBm it gives up.

```cpp
do { ... rsrpThreshold += 3; ... }
while (remainingCandidates.size () < (GetResourcePercentage () / 100.0) * mTotal);
```

### 5. Picking from the candidates

The scheduler chooses, which is where the pluggable design shows. In `NrSlUeMacSchedulerFixedMcs::DoNrSlAllocation`:

```cpp
// blind retransmission corresponds to HARQ enabled AND (PSFCH period == 0)
if (allocationInfo.m_harqEnabled && (GetMac ()->GetPsfchPeriod () == 0))
    selectedTxOpps = SelectResourcesForBlindRetransmissions (candResources);
else
    selectedTxOpps = SelectResourcesWithConstraint (candResources, allocationInfo.m_harqEnabled);
```

Both draw at random. The constrained version also keeps the minimum time gap to PSFCH, so a NACK can arrive before the next copy.

### 6. Grant and transmission

`CreateSpsGrant` or `CreateSinglePduGrant` stores the selected slots. Each slot gets SCI 1-A when a new reservation window starts, and otherwise only SCI 2. `slotNumInd` tells how many future slots it announces.

With PSFCH, a successful ACK cancels the remaining copies, handled through `DoReceivePsfch` and `NrSlUeMacHarq`.

## Side by side, step by step

| Step | 5G-LENA v2x | MoReV2X |
|---|---|---|
| Sensing on or off | `EnableSensing`, off by default | Always on, `RandomV2VSelection` is not implemented |
| Sensing window T0 | Pool, 100 or 1100 ms | 1100 ms, hardcoded |
| T1 | Attribute, checked against `T_proc,1` | 2 slots, hardcoded |
| T2 | `min(PDB, T2)` | `PDB - 1 slot` |
| Slot filter | TDD pattern and pool bitmap | None |
| Own past transmissions | Excluded, undone below X % | Excluded, undone only if **nothing** is left |
| X % | `ResourcePercentage`, attribute | 20 %, hardcoded |
| Threshold | One value, `SlThresPsschRsrp`, attribute | One value, `RSRPthreshold`, attribute |
| Priority in the threshold | No | No |
| Threshold loop | +3 dB, gives up at 0 dBm | +3 dB, no upper limit |
| Whole-slot exclusion | `WholeSlotExclusion` option | No |
| Final pick | Scheduler, uniform | Uniform |
| `Cresel` | `slResoReselCounter`, and `cResel = 10 * counter` in step 6 | `Cresel`, used directly in step 1 |
| Re-transmissions | Up to `slMaxTxTransNumPssch`, with PSFCH or blind | One blind |
| Re-evaluation | Not implemented. The code says so for dynamic grants | At `T3`, with `--AllSlots` and `--UMH` |
| Pre-emption | No | No |

Three differences change results:

- **Step 5 differs.** 5G-LENA drops the whole exclusion if it would go below X %, and counts X % of all candidates. MoReV2X keeps the exclusion unless the list is empty, and counts X % of what is left after it. So MoReV2X can end with a smaller final list.
- **`cResel` scale.** 5G-LENA checks 10 times the reselection counter in step 6, and MoReV2X checks `Cresel` periods, so the two look at a different number of future periods.
- **T0 and T1 defaults.** The example uses a 100 ms sensing window, MoReV2X 1100 ms. For RRIs up to 100 ms they see the same reservations, but a longer RRI needs a longer T0.

## Tuning, mapped

| What you want | 5G-LENA v2x | MoReV2X |
|---|---|---|
| RSRP threshold | `NrSlUeMac::SlThresPsschRsrp` | `--RSRP` |
| X % | `NrSlUeMac::ResourcePercentage` | Edit `m_sizeThreshold` |
| Keep probability | `slProbResourceKeep` | `--pkeep` |
| Selection window | `T1`, `T2`, `SlSelectionWindow` | `--PDB` |
| Sensing window | `SetSlSensingWindow` | Edit `m_sensingWindow` |
| MCS | Scheduler `Mcs` | `--mcs` |
| Subchannels | `SetSlSubchannelSize` | `--SubChannel` |
| Re-transmissions | `slMaxTxTransNumPssch`, PSFCH period | `--ReTx` |
| Different scheduler per UE | `SetNrSlSchedulerTypeId`, a real subclass | `--SchedulerMap`, flags |
| Selection rule | Subclass `NrSlUeMacSchedulerFixedMcs` | Edit the draw in `V2XSelectResources` |

To change the rule: 5G-LENA needs a subclass, but gives a clean per-UE scheduler. MoReV2X needs one line, in a function of 480 lines.

## Which one for what

| Question | Use |
|---|---|
| Re-evaluation, as it does not exist in 5G-LENA | MoReV2X |
| Mode 2 studies with a simple channel | MoReV2X |
| Groupcast or unicast, HARQ feedback, PSFCH | 5G-LENA |
| Channel realism, beams, antennas, TDD | 5G-LENA |
| New schedulers with a clean interface | 5G-LENA |

Numbers from one do not transfer directly to the other, because of the differences above, and because their channel and error models differ.

On the migration branch, both can live on the same base. Comparing them needs identical traffic and placement, which `--mobilityTrace` and your SUMO scenarios can provide, and the same `Tproc`, T0, `Cresel` and X settings.

## Your local patch

`patches/nr/local.patch` changes `nr-sl-ue-mac`, its scheduler, HARQ and the spectrum PHY, about 2,700 added lines. From its public methods, it adds relay behaviour for groupcast: helper UEs (`SetHelperId`, `ForceHarqIdForRelay`, `SelectOwnRelayResources`), and sharing of sensing (`SetSensingPeer`, `EvaluateSensedExclusion`). None of the upstream behaviour above is meant to change, but this notebook was written from the pinned commit, not from the patched files.